# STAT 940 Data Challenge 1: Ensemble of ImageNet-Pretrained CNNs

Fine-tunes ImageNet-pretrained ResNet-50 and VGG-16 models (loaded from the Hugging Face Hub with `timm`) on the
challenge's 32×32 images. Their predicted probabilities are then averaged (soft voting).
Competition: [stat940f23dc1 on Kaggle](https://www.kaggle.com/competitions/stat940f23dc1).

Runs locally on Linux CUDA GPU. It uses the same 90/10 train/validation split (seed 42)
as the ResNet notebook, so the validation accuracies can be compared directly.

## 1. Download the data

Requires a Kaggle API token at `~/.kaggle/kaggle.json` (Kaggle → Settings → API → *Create New Token*), and you must have joined the competition. The download is skipped if the data is already there.

In [1]:
import os

if not os.path.isdir("stat940f23dc1"):
    !kaggle competitions download -c stat940f23dc1
    !unzip -q stat940f23dc1.zip -x "__MACOSX/*"

100%|█████████████████████████████████████| 53.9M/53.9M [00:00<00:00, 96.7MB/s]

caution: excluded filename not matched:  __MACOSX/*


## 2. Imports and settings

Each model gets its own learning rate. To add a model to the ensemble, add a line to `MODELS` with any [timm checkpoint on the Hugging Face Hub](https://huggingface.co/timm).

In [2]:
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import get_cosine_schedule_with_warmup, set_seed

SEED = 42
set_seed(SEED)

DATA_DIR = "stat940f23dc1"
N_TRAIN, N_TEST, N_CLASSES = 50_000, 10_000, 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"

MODELS = {
    "ResNet-50": {"checkpoint": "resnet50.a1_in1k", "lr": 1e-4},
    "VGG-16":    {"checkpoint": "vgg16_bn.tv_in1k", "lr": 5e-5},
}

IMAGE_SIZE = 224
EPOCHS = 5
BATCH_SIZE = 64
WEIGHT_DECAY = 0.05
WARMUP_FRACTION = 0.1
NUM_WORKERS = 2

/home/u-waterloo/Documents/STAT-940---Deep-Learning-Data-Challenges-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load the images and split the data

In [3]:
def load_images(folder, n):
    return np.stack([np.array(Image.open(f"{folder}/{i}.jpg").convert("RGB")) for i in range(n)])

x_test = load_images(f"test/test", N_TEST)
x_all = load_images(f"train/train", N_TRAIN)
y_all = pd.read_csv(f"train_labels.csv").iloc[:, 1].to_numpy()

x_train, x_valid, y_train, y_valid = train_test_split(
    x_all, y_all, test_size=0.1, random_state=SEED
)
print(x_train.shape, x_valid.shape, x_test.shape)

(45000, 32, 32, 3) (5000, 32, 32, 3) (10000, 32, 32, 3)


## 4. Dataset and preprocessing

Images are resized to 224×224 and normalized with the ImageNet statistics each model was trained with. Training images also get random horizontal flips and small rotations.

In [4]:
class ImageDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = self.transform(Image.fromarray(self.images[i]))
        if self.labels is None:
            return image
        return image, int(self.labels[i])

def make_transform(data_config, train):
    steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE),
                               interpolation=transforms.InterpolationMode.BICUBIC)]
    if train:
        steps += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(5)]
    steps += [transforms.ToTensor(),
              transforms.Normalize(data_config["mean"], data_config["std"])]
    return transforms.Compose(steps)

def make_loader(images, labels, transform, shuffle):
    return DataLoader(ImageDataset(images, labels, transform), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=USE_AMP)

## 5. Fine-tuning

Each model's 1000-class ImageNet classifier is replaced with a new 10-class one. The whole network is then fine-tuned with AdamW, a cosine learning-rate schedule with warm-up, and mixed precision. The weights from the epoch with the best validation accuracy are kept.

In [5]:
@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        images = batch[0] if isinstance(batch, (list, tuple)) else batch
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
            logits = model(images.to(DEVICE))
        probs.append(logits.float().softmax(dim=-1).cpu())
    return torch.cat(probs).numpy()

def fine_tune(name, checkpoint, lr):
    model = timm.create_model(checkpoint, pretrained=True, num_classes=N_CLASSES).to(DEVICE)
    data_config = timm.data.resolve_model_data_config(model)

    train_loader = make_loader(x_train, y_train, make_transform(data_config, train=True), shuffle=True)
    valid_loader = make_loader(x_valid, y_valid, make_transform(data_config, train=False), shuffle=False)
    test_loader = make_loader(x_test, None, make_transform(data_config, train=False), shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_FRACTION * total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)

    best_acc, best_path = -1.0, f"best_{name.replace('/', '_')}.pt"
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for images, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                logits = model(images.to(DEVICE))
                loss = F.cross_entropy(logits, labels.to(DEVICE))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        valid_acc = (predict_proba(model, valid_loader).argmax(axis=1) == y_valid).mean()
        print(f"{name} | epoch {epoch}/{EPOCHS} | validation accuracy {valid_acc:.4f}")
        if valid_acc > best_acc:
            best_acc = valid_acc
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    valid_probs, test_probs = predict_proba(model, valid_loader), predict_proba(model, test_loader)

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return valid_probs, test_probs

## 6. Train the models

Validation and test probabilities are saved to disk.

In [6]:
valid_probs, test_probs = {}, {}
for name, cfg in MODELS.items():
    valid_probs[name], test_probs[name] = fine_tune(name, cfg["checkpoint"], cfg["lr"])
    np.save(f"valid_probs_{name.replace('/', '_')}.npy", valid_probs[name])
    np.save(f"test_probs_{name.replace('/', '_')}.npy", test_probs[name])

ResNet-50 | epoch 1/5 | validation accuracy 0.8454
ResNet-50 | epoch 2/5 | validation accuracy 0.8960
ResNet-50 | epoch 3/5 | validation accuracy 0.9150
ResNet-50 | epoch 4/5 | validation accuracy 0.9198
ResNet-50 | epoch 5/5 | validation accuracy 0.9200
VGG-16 | epoch 1/5 | validation accuracy 0.8814
VGG-16 | epoch 2/5 | validation accuracy 0.9134
VGG-16 | epoch 3/5 | validation accuracy 0.9304
VGG-16 | epoch 4/5 | validation accuracy 0.9404
VGG-16 | epoch 5/5 | validation accuracy 0.9450


## 7. Ensemble (soft voting)

The models' class probabilities are averaged, and the class with the highest average is predicted.

In [7]:
ensemble_valid = np.mean(list(valid_probs.values()), axis=0)
ensemble_test = np.mean(list(test_probs.values()), axis=0)

results = {name: (p.argmax(axis=1) == y_valid).mean() for name, p in valid_probs.items()}
results["Ensemble (soft voting)"] = (ensemble_valid.argmax(axis=1) == y_valid).mean()

results_table = pd.DataFrame({"Validation accuracy": results})
results_table

,Validation accuracy
ResNet-50,0.9200
VGG-16,0.9450
Ensemble (soft voting),0.9508


## 8. Write the submission

In [8]:
submission = pd.DataFrame({"id": np.arange(len(ensemble_test)), "label": ensemble_test.argmax(axis=1)})
submission.to_csv("predictions_ensemble.csv", index=False)
submission.head()

,id,label
0,0,4
1,1,1
2,2,2
3,3,3
4,4,9
